In [15]:
# This file compares new implementations of Algorithm 13,
# with Reviewer t5C9's code.
# To reproduce the results, please use the HIGH-RAM CPU of google colab,
# which has 8 CPU cores and 50.99 GB of RAM.


In [16]:
!apt-get update
!apt-get install -y libtbb-dev

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (4,372 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libtbb-dev is already the newest version (2021.5.0

In [17]:
%%writefile Variant1_new_implementation.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <stack>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

// ============================================================
// GRAPH
// ============================================================

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double,int>,
        vector<pair<double,int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<int>& visited,
    int token,
    vector<int>& nodes)
{
    nodes.clear();

    stack<int> st;

    st.push(start);

    visited[start] = token;

    while (!st.empty()) {

        int u = st.top();
        st.pop();

        nodes.push_back(u);

        for (const auto& e : graph[u]) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                st.push(v);
            }
        }
    }
}

// ============================================================
// MAIN THREAD
// ============================================================

void main_thread_func(
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // initially all active
    vector<char> active(num_edges, 1);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop_front();
        }

        // remove edge task
        active[task] = 0;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    vector<char> active(num_edges);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    int current_added = num_edges;

    int task;

    fill(active.begin(), active.end(), 0);

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.back();
            task_queue.pop_back();
        }

        if (task < num_edges - 1){
        for (int i = current_added - 1; i >= task + 1; --i) {
            active[i] = 1;
        }
        }
        current_added = task + 1;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{


    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    // ========================================================
    // TASK QUEUE
    // ========================================================

    deque<int> task_queue;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push_back(i);

    mutex queue_mutex;

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        edge_list.emplace_back(
            min(i, parent[i]),
            max(i, parent[i]),
            distance_matrix[i][parent[i]]);
    }

    sort(
        edge_list.begin(),
        edge_list.end(),
        [](const Edge& a, const Edge& b) {

            return get<2>(a) > get<2>(b);
        });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int,int>> edge_nodes;

    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);

        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex));
    }

    // ========================================================
    // TIMING
    // ========================================================



    main_thread_func(
        graph,
        edge_nodes,
        edge_weights,
        mmj_matrix,
        task_queue,
        queue_mutex);

    for (auto& th : threads)
        th.join();



    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

vector<vector<double>> createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    vector<vector<double>> A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}


// ============================================================
// MAIN
// ============================================================

int main() {

    int N =38999;

    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 7875;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs);

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30; i < row.size(); ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}



Overwriting Variant1_new_implementation.cpp


In [18]:
%%writefile Variant2_new_implementation.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>
#include <cstdint>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double, int>,
        vector<pair<double, int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<uint32_t>& visited,
    uint32_t token,
    vector<int>& nodes,
    vector<int>& stack_buffer)
{
    nodes.clear();

    stack_buffer.clear();

    stack_buffer.push_back(start);

    visited[start] = token;

    while (!stack_buffer.empty()) {

        int u = stack_buffer.back();
        stack_buffer.pop_back();

        nodes.push_back(u);

        const auto& neighbors = graph[u];

        for (const auto& e : neighbors) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                stack_buffer.push_back(v);
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int, int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    queue<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // active edges
    vector<char> active(num_edges, 1);

    // IMPORTANT:
    // each worker must progressively remove edges
    int current_removed = -1;

    vector<uint32_t> visited(n, 0);

    uint32_t token = 1;

    vector<int> tree1;
    vector<int> tree2;
    vector<int> stack_buffer;

    tree1.reserve(n);
    tree2.reserve(n);
    stack_buffer.reserve(n);

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop();
        }

        // ====================================================
        // cumulative edge removals
        // ====================================================

        for (int i = current_removed + 1; i <= task; ++i)
            active[i] = 0;

        current_removed = task;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        // ====================================================
        // DFS 1
        // ====================================================

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1,
            stack_buffer
        );

        // ====================================================
        // DFS 2
        // ====================================================

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2,
            stack_buffer
        );

        // ====================================================
        // fill MMJ matrix
        // ====================================================

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MAIN MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{
    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    queue<int> task_queue;

    mutex queue_mutex;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push(i);

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    // ========================================================
    // EDGE LIST
    // ========================================================

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        int p = parent[i];

        edge_list.emplace_back(
            min(i, p),
            max(i, p),
            distance_matrix[i][p]
        );
    }

    // descending order
    sort(edge_list.begin(),
         edge_list.end(),
         [](const Edge& a, const Edge& b) {
             return get<2>(a) > get<2>(b);
         });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int, int>> edge_nodes;
    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);
        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // IMPORTANT:
    // must build AFTER sorting edge_list
    // so edge IDs match task IDs
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex)
        );
    }

    for (auto& th : threads)
        th.join();

    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

Matrix createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    Matrix A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}

// ============================================================
// MAIN
// ============================================================

int main() {

    // WARNING:
    // Dense matrices explode in memory quickly

    int N = 38999;

    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 7875;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix =
        createDistanceMatrix(
            N,
            random_seed
        );

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs
        );

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed
         << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30;
         i < row.size();
         ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}


Overwriting Variant2_new_implementation.cpp


In [19]:
%%writefile Reviewer_t5C9_code_cpp_version.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <random>
#include <chrono>
#include <algorithm>
#include <list>
#include <iomanip>



using namespace std;

struct Edge {
    int u, v;
    double weight;
};

// Prim's MST (dense graph version)
vector<Edge> prim_mst(const vector<vector<double>>& D) {
    int n = D.size();
    vector<bool> in_mst(n, false);
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    key[0] = 0.0;

    for (int count = 0; count < n; ++count) {
        double min_val = numeric_limits<double>::infinity();
        int u = -1;
        for (int i = 0; i < n; ++i)
            if (!in_mst[i] && key[i] < min_val)
                min_val = key[i], u = i;

        if (u == -1) break;

        in_mst[u] = true;
        for (int v = 0; v < n; ++v)
            if (!in_mst[v] && D[u][v] < key[v])
                key[v] = D[u][v], parent[v] = u;
    }

    vector<Edge> edges;
    for (int v = 1; v < n; ++v)
        edges.push_back({parent[v], v, D[parent[v]][v]});
    return edges;
}

// Build CSR-like structure
void build_csr(int n, const vector<Edge>& mst_edges,
               vector<int>& ptr, vector<int>& adj_edges, vector<double>& adj_weights) {
    vector<int> edge_counts(n, 0);
    for (const auto& e : mst_edges) {
        edge_counts[e.u]++;
        edge_counts[e.v]++;
    }

    ptr.resize(n + 1);
    for (int i = 1; i <= n; ++i)
        ptr[i] = ptr[i - 1] + edge_counts[i - 1];

    adj_edges.resize(ptr[n]);
    adj_weights.resize(ptr[n]);
    vector<int> positions(n, 0);

    for (const auto& e : mst_edges) {
        for (int i = 0; i < 2; ++i) {
            int u = (i == 0) ? e.u : e.v;
            int v = (i == 0) ? e.v : e.u;
            int idx = ptr[u] + positions[u]++;
            adj_edges[idx] = v;
            adj_weights[idx] = e.weight;
        }
    }
}


#include <tbb/parallel_for.h>
#include <tbb/blocked_range.h>
#include <tbb/parallel_for_each.h>

vector<vector<double>> compute_bottleneck_matrix(int n,
    const vector<int>& ptr, const vector<int>& adj_edges, const vector<double>& adj_weights) {

    vector<vector<double>> bottleneck(n, vector<double>(n, 0.0));

    tbb::parallel_for(tbb::blocked_range<int>(0, n),
        [&](const tbb::blocked_range<int>& r) {
            for (int src = r.begin(); src < r.end(); ++src) {
                vector<bool> visited(n, false);
                vector<double> max_edges(n, 0.0);
                queue<pair<int, double>> q;

                visited[src] = true;
                q.push({src, 0.0});

                while (!q.empty()) {
                    auto [u, curr_max] = q.front(); q.pop();

                    for (int i = ptr[u]; i < ptr[u + 1]; ++i) {
                        int v = adj_edges[i];
                        double weight = adj_weights[i];

                        if (!visited[v]) {
                            double new_max = max(curr_max, weight);
                            visited[v] = true;
                            max_edges[v] = new_max;
                            q.push({v, new_max});
                        }
                    }
                }

                bottleneck[src] = move(max_edges); // safe: each src owns its row
            }
        }
    );

    return bottleneck;
}


vector<vector<double>> ultra_fast_wide(const vector<vector<double>>& D) {
    int n = D.size();


    vector<Edge> mst = prim_mst(D);

    vector<int> ptr, adj_edges;
    vector<double> adj_weights;
    build_csr(n, mst, ptr, adj_edges, adj_weights);

    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights);
}

vector<vector<double>> create_symmetric_distance_matrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}



int main() {


    int N =38999;
    int n_jobs = thread::hardware_concurrency();
    int random_seed = 7875;


    cout << "Number of nodes: "<< N << endl;
    cout << "Number of CPU cores: " << n_jobs << endl;



    auto distanceMatrix = create_symmetric_distance_matrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = ultra_fast_wide(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);
    cout << "Time used for MMJ matrix (Reviewer t5C9's code cpp version): " << time_used << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;

}



Overwriting Reviewer_t5C9_code_cpp_version.cpp


In [20]:
import numpy as np
import numba
import time
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances



@numba.njit(cache=True, fastmath=True)
def prim_mst(D):
    """
    Optimized Prim's MST using Numba JIT for dense graphs.
    Returns edges as list of (u, v, weight) tuples.
    """
    n = D.shape[0]
    in_mst = np.zeros(n, dtype=np.bool_)
    parent = np.full(n, -1, dtype=np.int64)
    key = np.full(n, np.inf, dtype=D.dtype)
    key[0] = 0.0

    for _ in range(n):
        # Find minimum key vertex not in MST
        u = -1
        min_val = np.inf
        for i in range(n):
            if not in_mst[i] and key[i] < min_val:
                min_val = key[i]
                u = i

        if u == -1:
            break

        in_mst[u] = True

        # Update neighbors
        for v in range(n):
            if not in_mst[v] and D[u, v] < key[v]:
                key[v] = D[u, v]
                parent[v] = u

    # Build edge list
    edges = []
    for v in range(1, n):
        u = parent[v]
        edges.append((u, v, D[u, v]))

    return edges

def build_csr_adjacency(n, mst_edges):
    """Convert MST to compressed sparse row (CSR) format"""

    edge_counts = np.zeros(n, dtype=np.int32)
    for u, v, _ in mst_edges:
        edge_counts[u] += 1
        edge_counts[v] += 1

    ptr = np.zeros(n+1, dtype=np.int32)
    ptr[1:] = np.cumsum(edge_counts)

    adj_edges = np.empty(ptr[-1], dtype=np.int32)
    adj_weights = np.empty(ptr[-1], dtype=np.float64)
    positions = np.zeros(n, dtype=np.int32)

    for u, v, w in mst_edges:
        for _ in range(2):  # Add both directions
            idx = ptr[u] + positions[u]
            adj_edges[idx] = v
            adj_weights[idx] = w
            positions[u] += 1
            u, v = v, u  # Swap for reverse direction

    return ptr, adj_edges, adj_weights

@numba.njit(parallel=True, cache=True, fastmath=True)
def compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights):
    """Numba-optimized BFS for all pairs bottleneck calculation"""
    bottleneck = np.zeros((n, n), dtype=np.float64)

    for src in numba.prange(n):
        visited = np.zeros(n, dtype=numba.boolean)
        max_edges = np.zeros(n, dtype=np.float64)
        queue = np.empty(n, dtype=np.int32)
        queue_weights = np.empty(n, dtype=np.float64)
        front = back = 0

        # Initialize BFS
        visited[src] = True
        queue[back] = src
        queue_weights[back] = 0.0
        back += 1

        while front < back:
            u = queue[front]
            current_max = queue_weights[front]
            front += 1

            # Process all neighbors
            start = ptr[u]
            end = ptr[u+1]
            for i in range(start, end):
                v = adj_edges[i]
                weight = adj_weights[i]

                if not visited[v]:
                    new_max = max(current_max, weight)
                    visited[v] = True
                    max_edges[v] = new_max
                    queue[back] = v
                    queue_weights[back] = new_max
                    back += 1

        bottleneck[src] = max_edges

    return bottleneck

def ultra_fast_wide(distance_matrix):
    n = distance_matrix.shape[0]
#     distance_matrix = np.round(pairwise_distances(X), 15)
    # mst = prim_mst(distance_matrix)  # Use Numba-optimized prim_mst

    start = time.time()
    mst = prim_mst(distance_matrix)
    end = time.time()
    time_used = end - start
    time_used = np.round(time_used, 3)

    # print(f"Time used for computing MST: {time_used}s" )


    # Convert MST to CSR format
    ptr, adj_edges, adj_weights = build_csr_adjacency(n, mst)

    # Compute bottleneck matrix with Numba
    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights)



In [21]:
import numpy as np
from numba import njit

@njit(cache=True)
def _mt19937_generate_pairs(seed: int, n_pairs: int):
    N_MT = 624
    M    = 397
    MATRIX_A   = np.uint32(0x9908b0df)
    UPPER_MASK = np.uint32(0x80000000)
    LOWER_MASK = np.uint32(0x7fffffff)

    # --- Init state (same formula as original, safe in uint32 under Numba) ---
    mt = np.zeros(N_MT, dtype=np.uint32)
    mt[0] = np.uint32(seed & 0xFFFFFFFF)
    for i in range(1, N_MT):
        prev = mt[i - 1]
        mt[i] = np.uint32(1812433253) * (prev ^ (prev >> np.uint32(30))) + np.uint32(i)

    index  = N_MT                              # triggers generate on first draw
    r0_arr = np.empty(n_pairs, dtype=np.uint32)
    r1_arr = np.empty(n_pairs, dtype=np.uint32)

    # --- Draw 2 × n_pairs uint32s ---
    for k in range(n_pairs * 2):

        # Twist when state is exhausted
        if index >= N_MT:
            for ii in range(N_MT):
                y = (mt[ii] & UPPER_MASK) | (mt[(ii + 1) % N_MT] & LOWER_MASK)
                mt[ii] = mt[(ii + M) % N_MT] ^ (y >> np.uint32(1))
                if y & np.uint32(1):
                    mt[ii] ^= MATRIX_A
            index = 0                          # reset inline — no nonlocal needed

        # Temper
        y = mt[index]
        index += 1
        y ^= (y >> np.uint32(11))
        y ^= (y << np.uint32(7))  & np.uint32(0x9d2c5680)
        y ^= (y << np.uint32(15)) & np.uint32(0xefc60000)
        y ^= (y >> np.uint32(18))

        if k & 1:
            r1_arr[k >> 1] = y   # odd draw → high bits
        else:
            r0_arr[k >> 1] = y   # even draw → low bits

    return r0_arr, r1_arr


def createDistanceMatrix_numba(N: int, seed: int) -> np.ndarray:
    n_pairs = N * (N - 1) // 2
    r0, r1 = _mt19937_generate_pairs(seed, n_pairs)

    # Replicate generate_canonical: r0/2^64 + r1/2^32
    canonical = (r0.astype(np.float64) / 18446744073709551616.0 +
                 r1.astype(np.float64) / 4294967296.0)

    lo, hi = 1.0, 19999.0
    vals = np.round((lo + (hi - lo) * canonical) * 100.0) / 100.0

    A = np.zeros((N, N), dtype=np.float64)
    i_idx, j_idx = np.triu_indices(N, k=1)
    A[i_idx, j_idx] = vals
    A[j_idx, i_idx] = vals
    return A

In [22]:
%%time

n =38999
random_seed = 7875

distance_matrix =  createDistanceMatrix_numba(n, random_seed)
distance_matrix =  np.array(distance_matrix)

CPU times: user 28.8 s, sys: 37.8 s, total: 1min 6s
Wall time: 1min 6s


In [23]:
# !pip install --upgrade tbb

In [24]:


print(f"Number of nodes: {n}" )


start = time.time()
mmj_matrix_Reviewer_t5C9_code = ultra_fast_wide(distance_matrix)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used for MMJ matrix (Reviewer t5C9's code python version): {time_used}s" )
print("Print last 30 values of the first row of MMJ matrix:")
print(mmj_matrix_Reviewer_t5C9_code[0, -30:])


Number of nodes: 38999
Time used for MMJ matrix (Reviewer t5C9's code python version): 12.923s
Print last 30 values of the first row of MMJ matrix:
[2.08 2.08 2.08 2.08 2.08 2.11 2.08 2.08 2.08 2.08 2.35 2.08 2.24 2.08
 2.1  2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 1.95 2.08
 2.12 2.53]


In [25]:
import gc
del distance_matrix
del mmj_matrix_Reviewer_t5C9_code
gc.collect()  # forces immediate collection



7

In [26]:
!g++ -std=c++17 -O3 -march=native Reviewer_t5C9_code_cpp_version.cpp  -o tt -ltbb
!./tt

Number of nodes: 38999
Number of CPU cores: 8
Time used for MMJ matrix (Reviewer t5C9's code cpp version): 19.024 seconds
Print last 30 values of the first row of mmj matrix: 
2.08 2.08 2.08 2.08 2.08 2.11 2.08 2.08 2.08 2.08 2.35 2.08 2.24 2.08 2.10 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 1.95 2.08 2.12 2.53 


In [27]:
!g++ -std=c++17 -O3 -march=native Variant1_new_implementation.cpp  -o tt -ltbb
!./tt

Number of nodes: 38999
Number of CPU cores: 8
Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):16.361 seconds
Print last 30 values of the first row of mmj matrix:
2.08 2.08 2.08 2.08 2.08 2.11 2.08 2.08 2.08 2.08 2.35 2.08 2.24 2.08 2.10 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 1.95 2.08 2.12 2.53 


In [28]:
!g++ -std=c++17 -O3 -march=native Variant2_new_implementation.cpp  -o tt -ltbb
!./tt

Number of nodes: 38999
Number of CPU cores: 8
Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):16.114 seconds
Print last 30 values of first row of mmj matrix:
2.08 2.08 2.08 2.08 2.08 2.11 2.08 2.08 2.08 2.08 2.35 2.08 2.24 2.08 2.10 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 2.08 1.95 2.08 2.12 2.53 


In [29]:
# Algorithm 13 appears to be slower than the Python version
# of Reviewer t5C9's code, but faster than its C++ version. However,
# Reviewer t5C9’s code/algorithm requires significantly more memory.
# For example, under the current 30 GB memory limit on Kaggle (as of July 2025):
# - The Python version of Reviewer t5C9’s code can handle graphs with up to ~36,000 nodes.
# - The C++ version can process graphs with up to ~38,000 nodes.
# - In contrast, Algorithm 13 can handle graphs with up to ~44,000 nodes.

In [30]:
import platform
import psutil

# CPU information
print("CPU Information:")
print(f"Processor: {platform.processor()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")

# RAM information
ram = psutil.virtual_memory()

print("\nRAM Information:")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

CPU Information:
Processor: x86_64
Physical cores: 4
Logical cores: 8
CPU frequency: 2250.00 MHz

RAM Information:
Total RAM: 50.99 GB
